# 01_data_prep.ipynb

## Setup

In [4]:
import os
import sys
from pathlib import Path

# Mount colab
if True:
    if not os.path.exists('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')

PROJECT_ROOT: Path = Path("/content/drive/MyDrive/python/projects/nlp_projects/med/biomedical-extraction-pipeline")
os.chdir(PROJECT_ROOT)

# Add the PROJECT_ROOT directory to the Python path to import modules
if str(PROJECT_ROOT) in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"ℹ️ Current Python version: {sys.version}")

ℹ️ Current Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [ ]:
# Install reqs (NOTE: restart after this)
if True:
    # Install the required packages
    %pip install -r {"requirements/01_data_prep.txt"}
   
    # # Upgrade IPython to ensure autoreload works correctly
    # %pip install Ipython --upgrade
    # %load_ext autoreload
    # %autoreload 2

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 11.3 MB/s eta 0:00:0000:01
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=efec912a983086725b28939735270ac9e20665b07598675439de1bcc60b4158e
  Stored in directory: /root/.cache/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf460593539805c3906722228
Successfully built docopt
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [7]:
# --- src ---
from src.config import load_configs
from src.utils import dynamic_module_reloader, seed_everything
from src.data_utils.processing import ensure_validation_split, flatten_bc5cdr

# --- other ---
import pandas as pd
import torch
from dotenv import load_dotenv
from pprint import pprint
from datasets import load_dataset, DatasetDict

MODULES_TO_RELOAD = [
    "src",
    "src.config",
    "src.utils",
    "src.data_utils",
    "src.data_utils.processing"
]

dynamic_module_reloader(modules=MODULES_TO_RELOAD, verbose=1)
config = load_configs()
load_dotenv()
seed_everything(config.env.seed)
pprint(config.to_dict())

# Constants
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

Reloaded: src
Reloaded: src.config
Reloaded: src.utils
Reloaded: src.data_utils
Reloaded: src.data_utils.processing
✅ Seed set to 42
{'data': {'data_folder': PosixPath('data/documents'),
          'dataset_name': 'bigbio/bc5cdr'},
 'env': {'seed': 42},
 'model': {'ner_model_checkpoint': 'microsoft/BiomedNLP-PubMedBERT-base-uncased',
           'ner_model_name': 'BiomedNLP-PubMedBERT-base-uncased-ner-abstract-bc5cdr'},
 'retrieval': {}}
Using device: cpu


In [5]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


## Data

### Download the dataset (bigbio/BC5CDR)

In [58]:
# Load the bigbio BC5CDR dataset
dataset = load_dataset(config.data.dataset_name, trust_remote_code=True, token=os.environ["HF_TOKEN"])
ensure_validation_split(dataset)

Dataset has a validation split already: no action performed.


DatasetDict({
    train: Dataset({
        features: ['passages'],
        num_rows: 500
    })
    test: Dataset({
        features: ['passages'],
        num_rows: 500
    })
    validation: Dataset({
        features: ['passages'],
        num_rows: 500
    })
})

In [160]:
# Show feature schema and a single example
print(f"Dataset shape: {dataset.shape}")
print("\nFeature schema (train):\n", dataset["train"].features)
print("\nTrain keys:", list(dataset["train"][0].keys()))
if False:
    print("\nExample :")
    pprint(dataset["train"][0]["passages"])

Dataset shape: {'train': (500, 1), 'test': (500, 1), 'validation': (500, 1)}

Feature schema (train):
 {'passages': [{'document_id': Value(dtype='string', id=None), 'type': Value(dtype='string', id=None), 'text': Value(dtype='string', id=None), 'entities': [{'id': Value(dtype='string', id=None), 'offsets': [[Value(dtype='int32', id=None)]], 'text': [Value(dtype='string', id=None)], 'type': Value(dtype='string', id=None), 'normalized': [{'db_name': Value(dtype='string', id=None), 'db_id': Value(dtype='string', id=None)}]}], 'relations': [{'id': Value(dtype='string', id=None), 'type': Value(dtype='string', id=None), 'arg1_id': Value(dtype='string', id=None), 'arg2_id': Value(dtype='string', id=None)}]}]}

Train keys: ['passages']


### Inspect dataset structure

In [56]:
# Expand a single record to see the nested structure
sample = dataset["train"][0]
passages_df = pd.json_normalize(sample['passages'])
print("Sample passages structure:")
display(passages_df.head())
print("\nColumns:", passages_df.columns.tolist())

# Flatten all passages from the entire train split
all_passages = []
for record in dataset["train"]:
    all_passages.extend(record['passages'])

passages_df_full = pd.json_normalize(all_passages)
print(f"\nFull passages DataFrame shape: {passages_df_full.shape}")
print("\nFirst few rows:")
display(passages_df_full.head())
print("\nData types:")
print(passages_df_full.dtypes)

Sample passages structure:


,document_id,type,text,entities,relations
0,227508,title,Naloxone reverses the antihypertensive effect ...,"[{'id': '0', 'offsets': [[0, 8]], 'text': ['Na...","[{'id': 'R0', 'type': 'CID', 'arg1_id': 'D0087..."
1,227508,abstract,"In unanesthetized, spontaneously hypertensive ...","[{'id': '2', 'offsets': [[93, 105]], 'text': [...","[{'id': 'R0', 'type': 'CID', 'arg1_id': 'D0087..."



Columns: ['document_id', 'type', 'text', 'entities', 'relations']

Full passages DataFrame shape: (1000, 5)

First few rows:


,document_id,type,text,entities,relations
0,227508,title,Naloxone reverses the antihypertensive effect ...,"[{'id': '0', 'offsets': [[0, 8]], 'text': ['Na...","[{'id': 'R0', 'type': 'CID', 'arg1_id': 'D0087..."
1,227508,abstract,"In unanesthetized, spontaneously hypertensive ...","[{'id': '2', 'offsets': [[93, 105]], 'text': [...","[{'id': 'R0', 'type': 'CID', 'arg1_id': 'D0087..."
2,354896,title,Lidocaine-induced cardiac asystole.,"[{'id': '0', 'offsets': [[0, 9]], 'text': ['Li...","[{'id': 'R0', 'type': 'CID', 'arg1_id': 'D0080..."
3,354896,abstract,Intravenous administration of a single 50-mg b...,"[{'id': '2', 'offsets': [[90, 99]], 'text': ['...","[{'id': 'R0', 'type': 'CID', 'arg1_id': 'D0080..."
4,435349,title,Suxamethonium infusion rate and observed fasci...,"[{'id': '0', 'offsets': [[0, 13]], 'text': ['S...","[{'id': 'R0', 'type': 'CID', 'arg1_id': 'D0133..."



Data types:
document_id    object
type           object
text           object
entities       object
relations      object
dtype: object


In [ ]:
if True:
    # Check the structure of entities and relations
    display(pd.json_normalize(passages_df_full['entities']))
    display(pd.json_normalize(passages_df_full['relations']))


,0,1,2,3,4,5,6,7,8,9,...,47,48,49,50,51,52,53,54,55,56
0,"{'id': '0', 'offsets': [[0, 8]], 'text': ['Nal...","{'id': '1', 'offsets': [[49, 58]], 'text': ['c...",None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,"{'id': '2', 'offsets': [[93, 105]], 'text': ['...","{'id': '3', 'offsets': [[181, 190]], 'text': [...","{'id': '4', 'offsets': [[244, 252]], 'text': [...","{'id': '5', 'offsets': [[274, 285]], 'text': [...","{'id': '6', 'offsets': [[306, 322]], 'text': [...","{'id': '7', 'offsets': [[354, 362]], 'text': [...","{'id': '8', 'offsets': [[364, 372]], 'text': [...","{'id': '9', 'offsets': [[469, 481]], 'text': [...","{'id': '10', 'offsets': [[487, 496]], 'text': ...","{'id': '11', 'offsets': [[563, 576]], 'text': ...",...,None,None,None,None,None,None,None,None,None,None
2,"{'id': '0', 'offsets': [[0, 9]], 'text': ['Lid...","{'id': '1', 'offsets': [[18, 34]], 'text': ['c...",None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,"{'id': '2', 'offsets': [[90, 99]], 'text': ['l...","{'id': '3', 'offsets': [[142, 152]], 'text': [...","{'id': '4', 'offsets': [[331, 347]], 'text': [...","{'id': '5', 'offsets': [[409, 418]], 'text': [...",None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,"{'id': '0', 'offsets': [[0, 13]], 'text': ['Su...","{'id': '1', 'offsets': [[41, 55]], 'text': ['f...",None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,"{'id': '2', 'offsets': [[143, 154]], 'text': [...","{'id': '3', 'offsets': [[163, 178]], 'text': [...","{'id': '4', 'offsets': [[558, 573]], 'text': [...","{'id': '5', 'offsets': [[614, 625]], 'text': [...","{'id': '6', 'offsets': [[799, 822]], 'text': [...","{'id': '7', 'offsets': [[838, 849]], 'text': [...","{'id': '8', 'offsets': [[858, 873]], 'text': [...","{'id': '9', 'offsets': [[994, 1001]], 'text': ...",None,None,...,None,None,None,None,None,None,None,None,None,None
996,"{'id': '0', 'offsets': [[45, 53]], 'text': ['m...",None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
997,"{'id': '1', 'offsets': [[145, 152]], 'text': [...","{'id': '2', 'offsets': [[165, 176]], 'text': [...","{'id': '3', 'offsets': [[181, 194]], 'text': [...","{'id': '4', 'offsets': [[212, 220]], 'text': [...","{'id': '5', 'offsets': [[281, 289]], 'text': [...","{'id': '6', 'offsets': [[356, 364]], 'text': [...","{'id': '7', 'offsets': [[412, 423]], 'text': [...","{'id': '8', 'offsets': [[447, 458]], 'text': [...","{'id': '9', 'offsets': [[467, 474]], 'text': [...","{'id': '10', 'offsets': [[512, 525]], 'text': ...",...,None,None,None,None,None,None,None,None,None,None
998,None,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,"{'id': 'R0', 'type': 'CID', 'arg1_id': 'D00875...",None,None,None,None,None,None,None,None,None,None,None,None,None
1,"{'id': 'R0', 'type': 'CID', 'arg1_id': 'D00875...",None,None,None,None,None,None,None,None,None,None,None,None,None
2,"{'id': 'R0', 'type': 'CID', 'arg1_id': 'D00801...",None,None,None,None,None,None,None,None,None,None,None,None,None
3,"{'id': 'R0', 'type': 'CID', 'arg1_id': 'D00801...",None,None,None,None,None,None,None,None,None,None,None,None,None
4,"{'id': 'R0', 'type': 'CID', 'arg1_id': 'D01339...",None,None,None,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,"{'id': 'R0', 'type': 'CID', 'arg1_id': 'D01398...","{'id': 'R1', 'type': 'CID', 'arg1_id': 'D01398...",None,None,None,None,None,None,None,None,None,None,None,None
996,"{'id': 'R0', 'type': 'CID', 'arg1_id': 'D01260...","{'id': 'R1', 'type': 'CID', 'arg1_id': 'D00351...",None,None,None,None,None,None,None,None,None,None,None,None
997,"{'id': 'R0', 'type': 'CID', 'arg1_id': 'D01260...","{'id': 'R1', 'type': 'CID', 'arg1_id': 'D00351...",None,None,None,None,None,None,None,None,None,None,None,None
998,"{'id': 'R0', 'type': 'CID', 'arg1_id': 'D00105...","{'id': 'R1', 'type': 'CID', 'arg1_id': 'D00105...",None,None,None,None,None,None,None,None,None,None,None,None


In [154]:
# Flatten the passages using the utility function across all the splits
flattened_dataset = DatasetDict({split: flatten_bc5cdr(dataset[split]) for split in dataset.keys()})
flattened_dataset

DatasetDict({
    train: Dataset({
        features: ['document_id', 'text', 'entities', 'relations'],
        num_rows: 500
    })
    test: Dataset({
        features: ['document_id', 'text', 'entities', 'relations'],
        num_rows: 500
    })
    validation: Dataset({
        features: ['document_id', 'text', 'entities', 'relations'],
        num_rows: 500
    })
})

### Save processed data

In [ ]:
# Save the flattened dataset to disk
flattened_dataset.save_to_disk("data/processed/bc5cdr_flattened_hf")

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

# Requirements (full snapshot)

In [8]:
!pip freeze > snapshots/01_data_prep.txt